# Facilitator-Member Difference Identification

## Creating the csv file for the following:

`Name-conference-year`

**Also for the following:**

- facilitator vs. non-facilitator

- ppl-team vs. ppl-not-team

- ppl-funded-team vs. ppl-not-funded-team

In [ ]:
# ==================== NICO person-level pipeline with ANNOTATIONS ====================
import json, re
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter

# ---- CONFIG ----
# DATA_DIR = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/data")
# OUTPUT_DIR = Path("/Users/maxchalekson/Desktop/outputs")
DATA_DIR = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/data")
OUTPUT_DIR = Path("/Users/maxchalekson/Desktop/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_PERSON_SESSION = []   # collect across conferences
ALL_PERSON_YEAR    = []   # collect across conferences

# ---- HELPERS ----
def _load_json(fp: Path):
    with open(fp, "r") as f:
        return json.load(f)

def _norm_code_name(name: str) -> str:
    """
    Normalize annotation code names to snake-case tokens for column names.
    E.g., "Knowledge Sharing" -> "knowledge_sharing"
          "Coordination and Decision Practices" -> "coordination_decision_practices"
    """
    s = name.strip().lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = "_".join(s.split())
    return s

# ---- LOAD CONFERENCE BUNDLE ----
def load_conference_data(conf_path: Path):
    conf_name   = conf_path.name   # e.g., "2021MZT"
    year        = int(conf_name[:4])
    conference  = conf_name[4:]

    outcome          = _load_json(conf_path / f"{conf_name}_outcome.json")
    person_to_team   = _load_json(conf_path / f"{conf_name}_person_to_team.json")
    session_outcomes = _load_json(conf_path / f"{conf_name}_session_outcomes.json")

    # features_*.json (per session)
    features = {}
    for fp in conf_path.glob("features_*.json"):
        sid = fp.stem.replace("features_", "")      # e.g., "2021_09_30_MZT_S5"
        features[sid] = _load_json(fp)

    return year, conference, outcome, person_to_team, session_outcomes, features

# ---- PERSON METRICS FROM TRANSCRIPTS + ANNOTATIONS (session_data/*.json) ----
def extract_person_metrics_from_session_data(conf_path: Path, session_id: str):
    """
    Returns dict: { person_name -> { p_* metrics, ann_* metrics } } for one session.
    - p_*: speaking, turns, interruptions, overlaps, screenshares, smiles, nods
    - ann_*: for each annotation code name:
        ann_<code>_count, ann_<code>_sum_score, ann_<code>_mean_score
    """
    session_file = conf_path / "session_data" / f"{session_id}.json"
    if not session_file.exists():
        return {}

    data = _load_json(session_file)
    rows = data.get("all_data", [])
    # Counters
    by_person_counts = defaultdict(lambda: Counter())  # p_* tallies
    # For annotations, we need per code: count, sum of scores -> then derive mean
    by_person_ann_count = defaultdict(lambda: Counter())
    by_person_ann_sum   = defaultdict(lambda: Counter())

    for r in rows:
        speaker = r.get("speaker")
        if not speaker:
            continue

        # --- p_* metrics ---
        dur = r.get("speaking_duration", 0) or 0
        by_person_counts[speaker]["p_speaking_duration_sec"] += float(dur)
        by_person_counts[speaker]["p_turns"] += 1

        if str(r.get("interuption", "")).strip().lower() == "yes":
            by_person_counts[speaker]["p_interruptions_made"] += 1
        if str(r.get("overlap", "")).strip().lower() == "yes":
            by_person_counts[speaker]["p_overlaps"] += 1
        if str(r.get("screenshare", "")).strip().lower() == "yes":
            by_person_counts[speaker]["p_screenshare_segments"] += 1

        by_person_counts[speaker]["p_smile_self_total"]  += float(r.get("smile_self", 0) or 0)
        by_person_counts[speaker]["p_smile_other_total"] += float(r.get("smile_other", 0) or 0)
        by_person_counts[speaker]["p_nods_received"]     += float(r.get("nods_others", 0) or 0)

        # --- ann_* metrics ---
        ann = r.get("annotations") or {}
        if isinstance(ann, dict):
            for raw_code, payload in ann.items():
                code = _norm_code_name(raw_code)
                # Count one utterance carrying this code
                by_person_ann_count[speaker][f"ann_{code}_count"] += 1
                # Pull "score" if present
                score = None
                if isinstance(payload, dict):
                    score = payload.get("score", None)
                if score is not None:
                    try:
                        s_val = float(score)
                    except Exception:
                        s_val = None
                    if s_val is not None:
                        by_person_ann_sum[speaker][f"ann_{code}_sum_score"] += s_val

    # Build person metrics dict (combine p_* and ann_*)
    out = {}
    for person in set(list(by_person_counts.keys()) + list(by_person_ann_count.keys())):
        d = dict(by_person_counts[person])

        # Merge ann counts + sums and compute means where possible
        for k, v in by_person_ann_count[person].items():
            d[k] = float(v)  # count
            # derive code token
            code = k.replace("ann_", "").replace("_count", "")
            sum_key = f"ann_{code}_sum_score"
            mean_key = f"ann_{code}_mean_score"
            ssum = float(by_person_ann_sum[person].get(sum_key, 0.0))
            d[sum_key] = ssum
            # mean only if count > 0 and we observed any score
            if v > 0 and ssum > 0:
                d[mean_key] = ssum / v
            else:
                # not all annotations have scores; keep NaN to avoid bias
                d[mean_key] = float("nan")
        out[person] = d

    return out

# ---- BUILD person-session rows ----
def build_person_session(year, conference, conf_path, session_outcomes, features):
    """One row per (person, session): role_in_session, ctx_* session features, p_* + ann_* metrics."""
    special = {"missing_names", "people_not_in_any_team"}
    sessions = [sid for sid in session_outcomes.keys() if sid not in special]
    rows = []

    for sid in sessions:
        so = session_outcomes[sid]
        facilitators = set(so.get("facilitators", []) or [])
        speakers     = set(so.get("all_speakers", []) or [])

        # union of all team members in 'teams'
        members = set()
        for _, tinfo in (so.get("teams", {}) or {}).items():
            for m in tinfo.get("members", []) or []:
                members.add(m)

        people = facilitators | speakers | members

        # session-level features (context -> ctx_*)
        ctx = {f"ctx_{k}": v for k, v in (features.get(sid, {}) or {}).items()}

        # person-level from transcripts + annotations
        person_metrics = extract_person_metrics_from_session_data(conf_path, sid)

        for person in sorted(people):
            if person in facilitators:
                role_in_session = "facilitator"
            elif person in members:
                role_in_session = "member"
            elif person in speakers:
                role_in_session = "participant"
            else:
                role_in_session = "unknown"

            pmet = person_metrics.get(person, {})  # p_* + ann_* keys
            rows.append({
                "person_name": person,
                "conference": conference,
                "year": year,
                "session_id": sid,
                "role_in_session": role_in_session,
                **ctx,
                **pmet
            })

    return pd.DataFrame(rows)

# ---- BUILD person-year (aggregate) ----
def build_person_year(person_session_df, person_to_team):
    """
    Aggregate to one row per (person, conference, year):
      - role tallies & primary role
      - mean of ctx_*, p_*, ann_* across sessions attended
      - team outcomes joined by person
    """
    if person_session_df.empty:
        return person_session_df

    # Role tallies
    pivot = (person_session_df
             .pivot_table(index=["person_name","conference","year"],
                          columns="role_in_session",
                          values="session_id",
                          aggfunc="nunique",
                          fill_value=0)
             .reset_index())
    for col in ["facilitator","member","participant","unknown"]:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot["sessions_total"] = pivot["facilitator"] + pivot["member"] + pivot["participant"] + pivot["unknown"]

    # Primary role: facilitator > member > participant > unknown
    role_priority = {"facilitator": 3, "member": 2, "participant": 1, "unknown": 0}
    def primary_role(row):
        # choose highest priority among any nonzero counts
        best = max(role_priority, key=lambda r: (row.get(r,0)>0, role_priority[r]))
        return best
    pivot["role_primary"] = pivot.apply(primary_role, axis=1)

    # Identify metric columns to average across sessions
    metric_cols = [c for c in person_session_df.columns if c.startswith("ctx_") or c.startswith("p_") or c.startswith("ann_")]
    if metric_cols:
        agg = (person_session_df
               .groupby(["person_name","conference","year"], as_index=False)[metric_cols]
               .mean(numeric_only=True))
        out = pivot.merge(agg, on=["person_name","conference","year"], how="left")
    else:
        out = pivot

    # Team outcomes from person_to_team (across the whole dataset; if needed per conf-year, adjust mapping)
    team_rows = []
    for pname in out["person_name"]:
        lst = person_to_team.get(pname, [])
        funded = sum(1 for t in lst if t.get("funded_status", 0) == 1)
        unfund = sum(1 for t in lst if t.get("funded_status", 0) == 0)
        team_rows.append((pname, funded, unfund, ", ".join([t.get("team_id","") for t in lst])))
    team_df = pd.DataFrame(team_rows, columns=["person_name","teams_funded","teams_unfunded","team_ids"])
    team_df["teams_total"] = team_df["teams_funded"] + team_df["teams_unfunded"]

    out = out.merge(team_df, on="person_name", how="left")

    return out

# ---- DRIVER: iterate conferences, build combined outputs ----
for conf_path in sorted(DATA_DIR.iterdir()):
    if not conf_path.is_dir():
        continue
    conf_name = conf_path.name
    core = conf_path / f"{conf_name}_session_outcomes.json"
    if not core.exists():
        continue

    year, conference, outcome, person_to_team, session_outcomes, features = load_conference_data(conf_path)

    ps = build_person_session(year, conference, conf_path, session_outcomes, features)
    py = build_person_year(ps, person_to_team)

    if not ps.empty:
        ALL_PERSON_SESSION.append(ps)
    if not py.empty:
        ALL_PERSON_YEAR.append(py)

# ---- COMBINE to single files ----
if ALL_PERSON_YEAR:
    all_py = (pd.concat(ALL_PERSON_YEAR, ignore_index=True)
                .sort_values(["person_name","year","conference"]))

    # --- add role flags for contrasts ---
    all_py["role_facilitator"]    = (all_py["role_primary"] == "facilitator").astype(int)
    all_py["role_nonfacilitator"] = 1 - all_py["role_facilitator"]

    all_py["teams_total"]  = all_py["teams_total"].fillna(0)
    all_py["teams_funded"] = all_py["teams_funded"].fillna(0)

    all_py["role_on_team"]   = (all_py["teams_total"]  > 0).astype(int)
    all_py["role_in_funded"] = (all_py["teams_funded"] > 0).astype(int)

    all_py["role_member"]      = (all_py["role_primary"] == "member").astype(int)
    all_py["role_participant"] = (all_py["role_primary"] == "participant").astype(int)

    out_path = OUTPUT_DIR / "ALL_person_year.csv"
    all_py.to_csv(out_path, index=False)
    print(f"Wrote ONE combined file: {out_path}  ({len(all_py)} rows)")
else:
    print("No person-year rows produced.")

# Optional: combined person-session file
# if ALL_PERSON_SESSION:
#     all_ps = (pd.concat(ALL_PERSON_SESSION, ignore_index=True)
#                 .sort_values(["person_name","year","conference","session_id"]))
#     out_path_ps = OUTPUT_DIR / "ALL_person_session.csv"
#     all_ps.to_csv(out_path_ps, index=False)
#     print(f"Wrote combined person-session file: {out_path_ps}  ({len(all_ps)} rows)")
# ==================== end ====================

## Regression Analysis

Figuring out the question: 

**By analyzing at the individual level, do facilitators act differently than team members?**

Also, considering again, the classification from above:

- facilitator vs. non-facilitator

- ppl-team vs. ppl-not-team

- ppl-funded-team vs. ppl-not-funded-team

# Running the features alone (without existing knowledge)

Determining if people are (or not) facilitator, funded-team, on-team.

**Probably would just run cross-validation on this.**

In [2]:
# ==================== ALL-IN-ONE: regressions + CV classification + graphs ====================
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# -------- CONFIG --------
CSV_PATH = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year.csv")
OUT_DIR  = Path("/Users/maxchalekson/Desktop/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

USE_CONTROLS = True          # add sessions_total + ctx_* to regressions
INCLUDE_CTX_IN_CV = True     # include ctx_* alongside p_* and ann_* in classifiers
RANDOM_STATE = 42
N_SPLITS = 5

# ==================== LOAD ====================
df = pd.read_csv(CSV_PATH)

# ==================== PART A: DESCRIPTIVE REGRESSIONS ====================
import statsmodels.formula.api as smf

def _is_num_and_nonempty(s: pd.Series) -> bool:
    return pd.api.types.is_numeric_dtype(s) and s.notna().any()

# Outcomes
p_outcomes          = [c for c in df.columns if c.startswith("p_") and _is_num_and_nonempty(df[c])]
ann_count_outcomes  = [c for c in df.columns if c.startswith("ann_") and c.endswith("_count") and _is_num_and_nonempty(df[c])]
ann_mean_outcomes   = [c for c in df.columns if c.startswith("ann_") and c.endswith("_mean_score") and _is_num_and_nonempty(df[c])]

role_flags = ["role_facilitator","role_on_team","role_in_funded"]
for r in role_flags:
    if r not in df.columns:
        raise ValueError(f"Missing role flag: {r}")

# Controls
controls = []
if USE_CONTROLS:
    if "sessions_total" in df.columns and _is_num_and_nonempty(df["sessions_total"]):
        controls.append("sessions_total")
    # add all numeric ctx_* as controls
    ctx_cols = [c for c in df.columns if c.startswith("ctx_") and _is_num_and_nonempty(df[c])]
    controls += ctx_cols

def build_formula(y, x, controls_list):
    rhs = [x] + (controls_list or [])
    return f"{y} ~ " + " + ".join(rhs)

reg_rows = []
summaries_txt = []

for contrast in role_flags:
    for outcome in (p_outcomes + ann_count_outcomes + ann_mean_outcomes):
        needed = [outcome, contrast] + controls
        d = df[needed].dropna().copy()
        if len(d) < 25:  # tiny samples are unstable; skip quietly
            continue
        fml = build_formula(outcome, contrast, controls)
        model = smf.ols(fml, data=d).fit()
        robust = model.get_robustcov_results(cov_type="HC3")

        # safe param extraction
        names  = robust.model.exog_names
        params = dict(zip(names, robust.params))
        ses    = dict(zip(names, robust.bse))
        pvals  = dict(zip(names, robust.pvalues))

        coef = params.get(contrast, np.nan)
        se   = ses.get(contrast, np.nan)
        pval = pvals.get(contrast, np.nan)

        reg_rows.append({
            "outcome": outcome,
            "contrast": contrast,
            "n_used": len(d),
            "controls": ", ".join(controls) if controls else "(none)",
            "coef_role": coef,
            "se_role": se,
            "p_role": pval,
            "r2": robust.rsquared,
            "r2_adj": robust.rsquared_adj,
            "formula": fml,
        })
        summaries_txt.append(f"=== {contrast} :: {outcome} ===\n{robust.summary().as_text()}\n")

# Save regression outputs
reg_df = pd.DataFrame(reg_rows).sort_values(["contrast","outcome"])
reg_df.to_csv(OUT_DIR / "regression_tidy_results.csv", index=False)
with open(OUT_DIR / "regression_model_summaries.txt", "w") as f:
    f.write("\n\n".join(summaries_txt))

# ==================== PART B: PREDICTION (CV Logistic Regression) ====================
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_curve, auc
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

TARGETS = [
    ("role_facilitator","Facilitator vs Non-facilitator"),
    ("role_on_team","On-team vs Not-on-team"),
    ("role_in_funded","Funded vs Unfunded"),
]

# Exclude leaky / ID columns from features
LEAKY_OR_ID = {
    "person_name","conference","year",
    "facilitator","member","participant","unknown","sessions_total",
    "role_primary","role_facilitator","role_nonfacilitator",
    "role_on_team","role_in_funded","role_member","role_participant",
    "teams_total","teams_funded","teams_unfunded","team_ids"
}

p_feats   = [c for c in df.columns if c.startswith("p_")   and _is_num_and_nonempty(df[c])]
ann_feats = [c for c in df.columns if c.startswith("ann_") and _is_num_and_nonempty(df[c])]
ctx_feats = [c for c in df.columns if c.startswith("ctx_") and _is_num_and_nonempty(df[c])] if INCLUDE_CTX_IN_CV else []
feat_cols = [c for c in (p_feats + ann_feats + ctx_feats) if c not in LEAKY_OR_ID]

# Drop zero-variance
feat_cols = [c for c in feat_cols if df[c].std(skipna=True) > 0]
if not feat_cols:
    raise ValueError("No usable behavior features (p_* / ann_* / ctx_*) after cleaning.")

pd.Series(feat_cols, name="feature").to_csv(OUT_DIR / "clf_features_used.csv", index=False)

num_pipe = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=0.0)),
                     ("scale", StandardScaler())])
pre = ColumnTransformer([("num", num_pipe, feat_cols)], remainder="drop")
logit = LogisticRegression(penalty="l2", solver="liblinear", class_weight="balanced",
                           max_iter=2000, random_state=RANDOM_STATE)
clf = Pipeline([("prep", pre), ("clf", logit)])
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

def plot_roc(y_true, y_prob, title, outpath):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    ax.plot([0,1],[0,1], linestyle="--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)
    return roc_auc

clf_rows = []
for target, desc in TARGETS:
    dsub = df.dropna(subset=[target]).copy()
    if dsub[target].nunique() < 2:
        continue
    y = dsub[target].astype(int)
    X = dsub[feat_cols]

    scoring = {"roc_auc":"roc_auc", "accuracy":"accuracy", "precision":"precision", "recall":"recall", "f1":"f1"}
    cv_res = cross_validate(clf, X, y, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False)
    metrics = {m: (cv_res[f"test_{m}"].mean(), cv_res[f"test_{m}"].std()) for m in scoring}

    # Held-out predictions for ROC + report
    y_prob = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:,1]
    y_hat  = (y_prob >= 0.5).astype(int)
    report = classification_report(y, y_hat, output_dict=True, zero_division=0)
    pd.DataFrame(report).to_csv(OUT_DIR / f"clf_classification_report_{target}.csv")

    roc_png = OUT_DIR / f"clf_roc_{target}.png"
    _ = plot_roc(y, y_prob, f"ROC – {desc}", roc_png)

    # Fit once on all data for coefficients
    clf.fit(X, y)
    coefs = pd.Series(clf.named_steps["clf"].coef_.ravel(), index=feat_cols).sort_values(ascending=False)
    coefs.to_csv(OUT_DIR / f"clf_top_coefs_{target}.csv")

    clf_rows.append({
        "target": target, "description": desc,
        "n_rows": int(len(y)), "pos_count": int(y.sum()), "neg_count": int((1-y).sum()),
        "roc_auc_mean": metrics["roc_auc"][0], "roc_auc_sd": metrics["roc_auc"][1],
        "accuracy_mean": metrics["accuracy"][0], "accuracy_sd": metrics["accuracy"][1],
        "precision_mean": metrics["precision"][0], "precision_sd": metrics["precision"][1],
        "recall_mean": metrics["recall"][0], "recall_sd": metrics["recall"][1],
        "f1_mean": metrics["f1"][0], "f1_sd": metrics["f1"][1],
        "roc_curve_png": str(roc_png)
    })

pd.DataFrame(clf_rows).sort_values("target").to_csv(OUT_DIR / "clf_results_summary.csv", index=False)

# ==================== PART C: GRAPHS (boxplots + mean±CI for key features) ====================
def mean_ci(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    n = len(s)
    if n == 0:
        return np.nan, np.nan
    m = s.mean()
    se = s.std(ddof=1)/np.sqrt(n) if n>1 else 0
    return m, 1.96*se

def subset_for_contrast(df0, contrast):
    # Keep it simple: no filtering; show all people for each contrast (0 vs 1)
    return df0.dropna(subset=[contrast])

labels = {
    "role_facilitator": ("Non-facilitator","Facilitator"),
    "role_on_team": ("Not on team","On team"),
    "role_in_funded": ("Unfunded team","Funded team"),
}

# Choose a small, interpretable set to plot (add more if you like)
plot_p = ["p_speaking_duration_sec","p_turns","p_interruptions_made","p_screenshare_segments"]
plot_ann = []
# Prefer common annotation codes if present
for cand in [
    "ann_knowledge_sharing_count","ann_coordination_and_decision_practices_count",
    "ann_coordination_decision_practices_count","ann_evaluation_practices_count",
    "ann_participation_dynamics_count","ann_relational_climate_count"
]:
    if cand in df.columns:
        plot_ann.append(cand)

def boxplot_two_groups(d, y, g, xlabels, fname):
    dd = d[[y,g]].dropna()
    if dd.empty: return
    data0 = dd.loc[dd[g]==0, y].values
    data1 = dd.loc[dd[g]==1, y].values
    fig, ax = plt.subplots(figsize=(6,4.5))
    ax.boxplot([data0, data1], labels=xlabels, showmeans=True)
    ax.set_title(f"{y} by {g}")
    ax.set_ylabel(y)
    ax.set_xlabel(g)
    ax.grid(True, linestyle="--", alpha=0.4)
    fig.tight_layout()
    fig.savefig(OUT_DIR / fname, dpi=200)
    plt.close(fig)

def bar_meanci_two_groups(d, y, g, xlabels, fname):
    dd = d[[y,g]].dropna()
    if dd.empty: return
    m0, c0 = mean_ci(dd.loc[dd[g]==0, y])
    m1, c1 = mean_ci(dd.loc[dd[g]==1, y])
    fig, ax = plt.subplots(figsize=(6,4.5))
    ax.bar([0,1], [m0,m1], yerr=[c0,c1], capsize=6)
    ax.set_xticks([0,1]); ax.set_xticklabels(xlabels)
    ax.set_title(f"Mean {y} (±95% CI) by {g}")
    ax.set_ylabel(y)
    ax.set_xlabel(g)
    ax.grid(True, linestyle="--", alpha=0.4, axis="y")
    fig.tight_layout()
    fig.savefig(OUT_DIR / fname, dpi=200)
    plt.close(fig)

for contrast in role_flags:
    dsub = subset_for_contrast(df, contrast)
    xlabels = labels[contrast]

    # p_*: use boxplots for continuous-ish, bar CI for counts
    if "p_speaking_duration_sec" in plot_p and "p_speaking_duration_sec" in df.columns:
        boxplot_two_groups(dsub, "p_speaking_duration_sec", contrast, xlabels,
                           f"box_p_speaking_duration_sec_{contrast}.png")
    if "p_turns" in plot_p and "p_turns" in df.columns:
        boxplot_two_groups(dsub, "p_turns", contrast, xlabels,
                           f"box_p_turns_{contrast}.png")
    for c in ["p_interruptions_made","p_screenshare_segments"]:
        if c in plot_p and c in df.columns:
            bar_meanci_two_groups(dsub, c, contrast, xlabels, f"bar_{c}_{contrast}.png")

    # ann_* counts: bar CI makes sense (means across people)
    for c in plot_ann:
        bar_meanci_two_groups(dsub, c, contrast, xlabels, f"bar_{c}_{contrast}.png")

print(f"\nDone. Outputs in: {OUT_DIR}")
print("Files written include:")
print(" - regression_tidy_results.csv")
print(" - regression_model_summaries.txt")
print(" - clf_results_summary.csv")
print(" - clf_features_used.csv")
print(" - clf_classification_report_<target>.csv")
print(" - clf_top_coefs_<target>.csv")
print(" - clf_roc_<target>.png")
print(" - box_/bar_*.png for key p_* and ann_* features across contrasts")
# ==================== END ====================

/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_42719/270766337.py:228: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([data0, data1], labels=xlabels, showmeans=True)
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_42719/270766337.py:228: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([data0, data1], labels=xlabels, showmeans=True)
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_42719/270766337.py:228: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([data0, data1], labels=xlabels, showmeans=True)
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_42


Done. Outputs in: /Users/maxchalekson/Desktop/outputs
Files written include:
 - regression_tidy_results.csv
 - regression_model_summaries.txt
 - clf_results_summary.csv
 - clf_features_used.csv
 - clf_classification_report_<target>.csv
 - clf_top_coefs_<target>.csv
 - clf_roc_<target>.png
 - box_/bar_*.png for key p_* and ann_* features across contrasts


## all_data_df aggregated by people

In [3]:
import pandas as pd
from pathlib import Path

IN_PATH  = Path("ALL_person_year.csv")  # update if needed
OUT_PATH = Path("ALL_person.csv")

df = pd.read_csv(IN_PATH)

# 1) Which columns to average vs sum vs max
mean_cols = [c for c in df.columns if c.startswith(("p_","ann_","ctx_"))]
sum_cols  = ["facilitator","member","participant","unknown",
             "sessions_total","teams_funded","teams_unfunded","teams_total"]
role_cols = ["role_facilitator","role_nonfacilitator","role_on_team",
             "role_in_funded","role_member","role_participant"]

# 2) Base aggregations
agg = {**{c:"mean" for c in mean_cols},
       **{c:"sum"  for c in sum_cols},
       **{c:"max"  for c in role_cols}}

# 3) Helpful extras (counts of appearances, n conferences/years, first/last year)
extras = df.groupby("person_name").agg(
    n_person_year_rows=("year","size"),
    n_conferences=("conference","nunique"),
    n_years=("year","nunique"),
    first_year=("year","min"),
    last_year=("year","max")
)

# 4) Unique team_ids concatenated
def _uniq_team_ids(series: pd.Series) -> str:
    vals = set()
    for x in series.dropna():
        for t in map(str.strip, str(x).split(",")):
            if t:
                vals.add(t)
    return ", ".join(sorted(vals)) if vals else ""

team_ids_unique = df.groupby("person_name")["team_ids"].apply(_uniq_team_ids).rename("team_ids_all")

# 5) Build the main aggregated frame
person_df = df.groupby("person_name").agg(agg).reset_index()

# 6) Primary role overall (by summed session counts; ties broken by priority)
role_priority = ["facilitator","member","participant","unknown"]
def primary_role_overall(row):
    scores = {r: row.get(r, 0) for r in role_priority}
    # pick the role with the highest summed sessions; use priority order for ties
    return max(role_priority, key=lambda r: (scores[r], role_priority.index(r)))

person_df["primary_role_overall"] = person_df.apply(primary_role_overall, axis=1)

# 7) Merge extras + team_ids list
person_df = (person_df
             .merge(extras, left_on="person_name", right_index=True, how="left")
             .merge(team_ids_unique, left_on="person_name", right_index=True, how="left"))

# 8) (Optional) Recompute overall role flags from primary_role_overall
person_df["is_facilitator_overall"] = (person_df["primary_role_overall"]=="facilitator").astype(int)
person_df["is_member_overall"]      = (person_df["primary_role_overall"]=="member").astype(int)
person_df["is_participant_overall"] = (person_df["primary_role_overall"]=="participant").astype(int)

# 9) Save
person_df.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH.resolve()}")

Saved: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person.csv


## fixing the regression table - visually

In [10]:
# Format regression_tidy_results.csv into publication-style tables with Coef (SE), p, and R²
import pandas as pd
import re
from pathlib import Path

# --- Paths (edit if needed) ---
IN_CSV  = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/results/regression_tidy_results.csv")           # your tidy results
OUT_DIR = Path("reg_tables")                            # where to save formatted tables
OUT_DIR.mkdir(exist_ok=True, parents=True)

# --- Load tidy results ---
df = pd.read_csv(IN_CSV)

# Expecting columns: outcome, contrast, n_used, controls, coef_role, se_role, p_role, r2, r2_adj, formula
# (these are what your generator wrote; if different, adjust the renames below)
tbl = df.rename(columns={
    "outcome":   "Outcome",
    "contrast":  "Model (Role Contrast)",
    "coef_role": "Coef",
    "se_role":   "SE",
    "p_role":    "p",
    "r2":        "R2",
    "r2_adj":    "R2_adj",
    "n_used":    "N"
})[["Model (Role Contrast)","Outcome","N","Coef","SE","p","R2","R2_adj","controls"]]

# --- Format Coef (SE) and p with significance stars ---
def pstars(p):
    if pd.isna(p): return ""
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return ""

tbl["Coef (SE)"] = tbl["Coef"].round(3).astype(str) + " (" + tbl["SE"].round(3).astype(str) + ")" + tbl["p"].apply(pstars)
tbl["p_fmt"] = tbl["p"].apply(lambda x: f"{x:.3g}" if pd.notna(x) else "")

# --- Final display columns ---
disp = tbl[["Model (Role Contrast)","Outcome","N","Coef (SE)","p_fmt","R2","R2_adj","controls"]].copy()
disp = disp.rename(columns={"p_fmt":"p","controls":"Controls"})

# Sort rows within each model by p-value (ascending), then by outcome
disp["p_for_sort"] = pd.to_numeric(df["p_role"], errors="coerce")
disp = disp.sort_values(["Model (Role Contrast)","p_for_sort","Outcome"]).drop(columns=["p_for_sort"])

# --- Save: one CSV + one Markdown per model contrast ---
for model_name, sub in disp.groupby("Model (Role Contrast)"):
    safe = re.sub(r"[^A-Za-z0-9_]+","_", model_name.strip())
    sub.to_csv(OUT_DIR / f"reg_table_{safe}.csv", index=False)
    sub.to_markdown(OUT_DIR / f"reg_table_{safe}.md", index=False)

# --- Save: one Excel workbook with a sheet per model ---
xlsx_path = OUT_DIR / "regression_tables_by_model.xlsx"
with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as xw:
    for model_name, sub in disp.groupby("Model (Role Contrast)"):
        sheet = model_name[:31]  # Excel sheet name limit
        sub.to_excel(xw, sheet_name=sheet, index=False)

print("Saved to:", OUT_DIR.resolve())
print("Workbook:", xlsx_path.resolve())

Saved to: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/reg_tables
Workbook: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/reg_tables/regression_tables_by_model.xlsx


## multicollinearity checking

In [13]:
# Candidate predictors (adjust list to whatever you expect to use)
candidates = [
    "p_speaking_duration_sec", "p_turns", "p_interruptions_made", "p_overlaps",
    "p_screenshare_segments", "p_smile_self_total", "p_smile_other_total", "p_nods_received",
    "ann_knowledge_sharing_mean_score", "ann_coordination_and_decision_practices_mean_score",
    "ann_relational_climate_mean_score", "ann_participation_dynamics_mean_score"
]

# keep only the ones that actually exist
X_feats = [c for c in candidates if c in df.columns]
controls = []
if "sessions_total" in df.columns: controls.append("sessions_total")
if "year" in df.columns:           controls.append("C(year)")
if "conference" in df.columns:     controls.append("C(conference)")

rhs = " + ".join(X_feats + controls)
formula = f"role_facilitator ~ {rhs}"
print("Formula:", formula)

# Drop rows with missing target/predictors
cols_needed = ["role_facilitator"] + X_feats + [c for c in controls if not c.startswith("C(")]
dd = df.dropna(subset=cols_needed).copy()

model = smf.logit(formula=formula, data=dd).fit(disp=False)
print(model.summary())
print("Pseudo R^2 (McFadden):", model.prsquared)

Formula: role_facilitator ~ p_speaking_duration_sec + p_turns + p_interruptions_made + p_overlaps + p_screenshare_segments + p_smile_self_total + p_smile_other_total + p_nods_received + ann_knowledge_sharing_mean_score + ann_coordination_and_decision_practices_mean_score + ann_relational_climate_mean_score + ann_participation_dynamics_mean_score + sessions_total + C(year) + C(conference)
                           Logit Regression Results                           
Dep. Variable:       role_facilitator   No. Observations:                   99
Model:                          Logit   Df Residuals:                       82
Method:                           MLE   Df Model:                           16
Date:                Fri, 26 Sep 2025   Pseudo R-squ.:                  0.5893
Time:                        12:11:24   Log-Likelihood:                -21.536
converged:                       True   LL-Null:                       -52.441
Covariance Type:            nonrobust   LLR p-value:    

In [14]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# Build design matrix for VIF using the same X_feats (no categorical terms here)
vif_df = dd[X_feats].fillna(0).copy()
vif_df = sm.add_constant(vif_df)
vif = []
for i, name in enumerate(vif_df.columns):
    vif.append((name, variance_inflation_factor(vif_df.values, i)))
vif_table = pd.DataFrame(vif, columns=["feature","VIF"]).sort_values("VIF", ascending=False)
print(vif_table)

                                              feature         VIF
0                                               const  202.387614
2                                             p_turns    3.763998
1                             p_speaking_duration_sec    3.576423
4                                          p_overlaps    3.374771
3                                p_interruptions_made    3.174510
7                                 p_smile_other_total    1.916842
6                                  p_smile_self_total    1.647009
8                                     p_nods_received    1.370500
11                  ann_relational_climate_mean_score    1.267252
9                    ann_knowledge_sharing_mean_score    1.222035
5                              p_screenshare_segments    1.134907
10  ann_coordination_and_decision_practices_mean_s...    1.124492
12              ann_participation_dynamics_mean_score    1.095659


In [15]:
drop_these = ["p_turns"]  # example if collinear with duration
X_feats = [c for c in X_feats if c not in drop_these]
rhs = " + ".join(X_feats + controls)
formula = f"role_facilitator ~ {rhs}"
model = smf.logit(formula=formula, data=dd).fit(disp=False)
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:       role_facilitator   No. Observations:                   99
Model:                          Logit   Df Residuals:                       83
Method:                           MLE   Df Model:                           15
Date:                Fri, 26 Sep 2025   Pseudo R-squ.:                  0.2806
Time:                        12:11:55   Log-Likelihood:                -37.727
converged:                       True   LL-Null:                       -52.441
Covariance Type:            nonrobust   LLR p-value:                   0.01416
                                                         coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------
Intercept                                             -4.2112      4.690     -0.898      0.369     -13.403       4.980
C(conferenc

### multicolinearity analysis summary



$R^2 \approx 0.28$
Strongest behavioral predictors of facilitation:
- speaking duration $(p=0.011)$ and self-smiling $(p=0.036)$

Other features (interruptions, overlaps, nods, annotation scores) did not reach sig level at $p < 0.05$.

Checking multicollinearity - all behavioral predictors within VIF ($<5$)



## Fine-tuning stuff (09/29/2025)

### floating points

In [4]:
import json, pandas as pd, numpy as np
from pathlib import Path
from collections import defaultdict, Counter

# ========= CONFIG =========
DATA_DIR   = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/data")  # root with 2021MZT, 2022SLU, ...
OUTPUT_DIR = Path("/Users/maxchalekson/Desktop/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WRITE_PERSON_SESSION = True  # set False to skip writing the audit csv

# ========= HELPERS =========
def _load_json(fp: Path):
    with open(fp, "r") as f:
        return json.load(f)

def extract_person_metrics_from_session_data(conf_path: Path, session_id: str):
    """
    Compute per-person p_* metrics for a single session from /session_data/<session_id>.json.
    Returns dict: person -> {p_*: value}
    """
    f = conf_path / "session_data" / f"{session_id}.json"
    if not f.exists():
        return {}

    data = _load_json(f)
    rows = data.get("all_data", [])
    by_person = defaultdict(lambda: Counter())
    for r in rows:
        speaker = r.get("speaker")
        if not speaker:
            continue
        dur = r.get("speaking_duration", 0) or 0
        by_person[speaker]["p_speaking_duration_sec"] += float(dur)
        by_person[speaker]["p_turns"] += 1

        if str(r.get("interuption","")).strip().lower() == "yes":
            by_person[speaker]["p_interruptions_made"] += 1
        if str(r.get("overlap","")).strip().lower() == "yes":
            by_person[speaker]["p_overlaps"] += 1
        if str(r.get("screenshare","")).strip().lower() == "yes":
            by_person[speaker]["p_screenshare_segments"] += 1

        by_person[speaker]["p_smile_self_total"]  += float(r.get("smile_self", 0) or 0)
        by_person[speaker]["p_smile_other_total"] += float(r.get("smile_other", 0) or 0)
        by_person[speaker]["p_nods_received"]     += float(r.get("nods_others", 0) or 0)

        # OPTIONAL: collect annotation counts/scores at utterance level if present
        ann = r.get("annotations") or {}
        for cat, info in ann.items():
            # normalize category name to snake-ish
            key = (
                cat.lower()
                   .replace("&","and")
                   .replace("/","_")
                   .replace(" ","_")
                   .replace("-", "_")
            )
            score = info.get("score", None)
            if score is not None:
                by_person[speaker][f"ann_{key}_count"] += 1
                by_person[speaker][f"ann_{key}_sum_score"] += float(score)

    # return dict of dicts
    return {p: dict(cnt) for p, cnt in by_person.items()}

def load_conference(conf_path: Path):
    conf_name = conf_path.name  # e.g., 2021MZT
    year = int(conf_name[:4])
    conference = conf_name[4:]

    session_outcomes = _load_json(conf_path / f"{conf_name}_session_outcomes.json")

    # features_*.json (session-level context; we’ll prefix ctx_*)
    features = {}
    for fp in conf_path.glob("features_*.json"):
        sid = fp.stem.replace("features_", "")
        features[sid] = _load_json(fp)

    return year, conference, session_outcomes, features

# ========= BUILD PERSON-SESSION =========
all_ps = []
for conf_path in sorted(DATA_DIR.iterdir()):
    if not conf_path.is_dir():
        continue
    conf_name = conf_path.name
    core = conf_path / f"{conf_name}_session_outcomes.json"
    if not core.exists():
        continue

    year, conference, session_outcomes, features = load_conference(conf_path)
    special = {"missing_names", "people_not_in_any_team"}
    sessions = [sid for sid in session_outcomes.keys() if sid not in special]

    for sid in sessions:
        so = session_outcomes[sid]
        facilitators = set(so.get("facilitators", []) or [])
        speakers     = set(so.get("all_speakers", []) or [])

        # team members from teams{}
        members = set()
        for _, tinfo in (so.get("teams", {}) or {}).items():
            for m in tinfo.get("members", []) or []:
                members.add(m)

        people = sorted(facilitators | speakers | members)
        ctx = {f"ctx_{k}": v for k, v in (features.get(sid, {}) or {}).items()}
        pmet = extract_person_metrics_from_session_data(conf_path, sid)

        for person in people:
            if person in facilitators:
                role_in_session = "facilitator"
            elif person in members:
                role_in_session = "member"
            elif person in speakers:
                role_in_session = "participant"
            else:
                role_in_session = "unknown"

            row = {
                "person_name": person,
                "conference": conference,
                "year": year,
                "session_id": sid,
                "role_in_session": role_in_session,
                **ctx
            }
            row.update(pmet.get(person, {}))
            all_ps.append(row)

ps_df = pd.DataFrame(all_ps)

# write person-session (audit)
if WRITE_PERSON_SESSION and not ps_df.empty:
    ps_out = OUTPUT_DIR / "ALL_person_session.csv"
    ps_df.to_csv(ps_out, index=False)
    print(f"Wrote person-session audit: {ps_out}  ({len(ps_df)} rows)")

# ========= AGGREGATE → PERSON-YEAR (SUM COUNTS; WEIGHTED MEANS FOR ann_*_mean_score) =========
if ps_df.empty:
    raise SystemExit("No person-session rows. Check DATA_DIR structure/files.")

keys = ["person_name","conference","year"]

# identify columns
p_cols = [c for c in ps_df.columns if c.startswith("p_")]
ctx_cols = [c for c in ps_df.columns if c.startswith("ctx_")]
ann_count_cols = [c for c in ps_df.columns if c.startswith("ann_") and c.endswith("_count")]
ann_sum_cols   = [c for c in ps_df.columns if c.startswith("ann_") and c.endswith("_sum_score")]

# base aggregations
agg_map = {}
# p_* sum (counts/segments) and duration sum
for c in p_cols:
    agg_map[c] = "sum"
# ctx_* mean across sessions
for c in ctx_cols:
    agg_map[c] = "mean"
# annotations: sum counts and sum_scores
for c in ann_count_cols + ann_sum_cols:
    agg_map[c] = "sum"

py_base = ps_df.groupby(keys, as_index=False).agg(agg_map)

# add sessions_total (unique sessions attended)
sesh_counts = (
    ps_df.groupby(keys, as_index=False)["session_id"]
         .nunique()
         .rename(columns={"session_id":"sessions_total"})
)
py = py_base.merge(sesh_counts, on=keys, how="left")

# derive role tallies per person-year from person-session
role_tallies = (
    ps_df.assign(one=1)
         .pivot_table(index=keys, columns="role_in_session", values="one", aggfunc="sum", fill_value=0)
         .reset_index()
         .rename(columns={"facilitator":"facilitator", "member":"member", "participant":"participant", "unknown":"unknown"})
)
# ensure all role columns exist
for col in ["facilitator","member","participant","unknown"]:
    if col not in role_tallies.columns:
        role_tallies[col] = 0
py = py.merge(role_tallies, on=keys, how="left")

# weighted means for each annotation mean_score = sum_score / count
ann_categories = set([c.replace("ann_","").replace("_count","") for c in ann_count_cols])
for cat in sorted(ann_categories):
    cnt_col = f"ann_{cat}_count"
    sum_col = f"ann_{cat}_sum_score"
    mean_col = f"ann_{cat}_mean_score"
    if cnt_col in py.columns and sum_col in py.columns:
        py[mean_col] = np.where(py[cnt_col].fillna(0) > 0,
                                pd.to_numeric(py[sum_col], errors="coerce") / pd.to_numeric(py[cnt_col], errors="coerce"),
                                np.nan)

# ===== dtype enforcement: ints for counts; floats for durations/scores =====
# counts (discrete)
count_like = [
    "p_turns","p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
    "sessions_total","facilitator","member","participant","unknown"
] + ann_count_cols

for c in count_like:
    if c in py.columns:
        py[c] = pd.to_numeric(py[c], errors="coerce").fillna(0).round().astype(int)

# sums of scores are numeric; they’ll typically be floats but are sums of discrete scores (keep as int if integral)
for c in ann_sum_cols:
    if c in py.columns:
        s = pd.to_numeric(py[c], errors="coerce").fillna(0)
        py[c] = np.where(np.isclose(s, np.round(s)), s.round().astype(int), s)  # keep int if whole, else float

# speaking duration is continuous; keep float (round for readability)
if "p_speaking_duration_sec" in py.columns:
    py["p_speaking_duration_sec"] = pd.to_numeric(py["p_speaking_duration_sec"], errors="coerce").round(3)

# ===== OPTIONAL: derive role flags =====
py["role_primary"] = py[["facilitator","member","participant","unknown"]].idxmax(axis=1)
py["role_facilitator"]   = (py["facilitator"]   > 0).astype(int)
py["role_member"]        = (py["member"]        > 0).astype(int)
py["role_participant"]   = (py["participant"]   > 0).astype(int)
py["role_nonfacilitator"]= (py["role_facilitator"] == 0).astype(int)

# If you have team files to compute on_team / funded, merge here (skipped because not loaded in this block)

# write person-year
py_out = OUTPUT_DIR / "ALL_person_year_FIXED.csv"
py.sort_values(["person_name","year","conference"]).to_csv(py_out, index=False)
print(f"Wrote person-year: {py_out}  ({len(py)} rows)")

# ========= AGGREGATE → PERSON (across all years) =========
sum_cols_person = [
    "p_turns","p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
    "sessions_total","facilitator","member","participant","unknown"
] + ann_count_cols + ann_sum_cols

mean_cols_person = [c for c in py.columns if c.startswith(("ctx_"))]  # context means (optional)
mean_cols_person += [c for c in py.columns if c.endswith("_mean_score")]  # annotation means

agg_person = {}
for c in sum_cols_person:
    if c in py.columns: agg_person[c] = "sum"
for c in mean_cols_person:
    if c in py.columns: agg_person[c] = "mean"

person = py.groupby("person_name", as_index=False).agg(agg_person)

# enforce ints again for counts
for c in sum_cols_person:
    if c in person.columns:
        person[c] = pd.to_numeric(person[c], errors="coerce").fillna(0).round().astype(int)

# keep durations and mean scores as floats (rounded)
for c in [x for x in person.columns if x.endswith("_mean_score") or x.startswith(("ctx_","p_speaking_duration_sec"))]:
    person[c] = pd.to_numeric(person[c], errors="coerce").round(3)

person_out = OUTPUT_DIR / "all_data_df-agg-ppl_FIXED.csv"
person.sort_values(["person_name"]).to_csv(person_out, index=False)
print(f"Wrote person-level: {person_out}  ({len(person)} rows)")

# ========= QUICK SANITY: show any columns that are float but expected int ========
def check_int_columns(df, cols, label):
    bad = []
    for c in cols:
        if c in df.columns and not pd.api.types.is_integer_dtype(df[c]):
            bad.append(c)
    if bad:
        print(f"[WARN] {label} columns not integer as expected:", bad)
    else:
        print(f"[OK] All {label} columns are integer.")

int_cols_py = ["p_turns","p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
               "sessions_total","facilitator","member","participant","unknown"] + ann_count_cols
check_int_columns(py, int_cols_py, "person-year count-like")

int_cols_person = [c for c in int_cols_py + ann_sum_cols if c in person.columns]
check_int_columns(person, int_cols_person, "person-level count-like")

Wrote person-session audit: /Users/maxchalekson/Desktop/outputs/ALL_person_session.csv  (2073 rows)
Wrote person-year: /Users/maxchalekson/Desktop/outputs/ALL_person_year_FIXED.csv  (790 rows)
Wrote person-level: /Users/maxchalekson/Desktop/outputs/all_data_df-agg-ppl_FIXED.csv  (670 rows)
[OK] All person-year count-like columns are integer.
[OK] All person-level count-like columns are integer.


### fixing the names

In [7]:
import re, unicodedata
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

# ========= PATHS =========
IN_DIR = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/")
PS_IN  = IN_DIR / "ALL_person_session.csv"
PY_IN  = IN_DIR / "ALL_person_year_FIXED.csv"
P_IN   = IN_DIR / "all_data_df-agg-ppl_FIXED.csv"

OUT_DIR = IN_DIR
MAP_OUT = OUT_DIR / "name_clean_map.csv"
PS_OUT  = OUT_DIR / "ALL_person_session_CLEANED.csv"
PY_OUT  = OUT_DIR / "ALL_person_year_FIXED_CLEANED.csv"
P_OUT   = OUT_DIR / "all_data_df-agg-ppl_FIXED_CLEANED.csv"

# ========= LOAD =========
ps = pd.read_csv(PS_IN)
py = pd.read_csv(PY_IN)
p  = pd.read_csv(P_IN)

# ========= NAME NORMALIZATION =========
DEGREE_SUFFIXES = [
    r"\bph\.?d\.?\b", r"\bmd\b", r"\bmd-ph\.?d\.?\b", r"\bscd\b", r"\bmsc\b", r"\bms\b",
    r"\bba\b", r"\bma\b", r"\bmba\b", r"\bdvm\b", r"\bdds\b"
]
ORG_DELIMS = [r"\s*-\s*", r"\s*—\s*", r"\s*–\s*", r"\s*\|\s*", r"\s*@\s*"]
TRAILERS   = [r"\(.*?\)", r"\[.*?\]", r"\{.*?\}"]

# If you know any manual fixes, put them here (raw or cleaned key -> canonical value)
alias_map = {
    # "Max Chalekson - UCLA": "Max Chalekson",
    # "Evey Huang, PhD": "Evey Huang",
}

def strip_accents(text: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))

def basic_name_clean(s: str) -> str:
    if not isinstance(s, str) or not s.strip():
        return ""
    x = s.strip()

    # remove emails
    x = re.sub(r"\b\S+@\S+\.\S+\b", " ", x)
    # drop anything in (), [], {}
    for patt in TRAILERS:
        x = re.sub(patt, " ", x)
    # drop degree suffixes
    for deg in DEGREE_SUFFIXES:
        x = re.sub(deg, " ", x, flags=re.IGNORECASE)
    # split off affiliation/org
    for delim in ORG_DELIMS:
        x = re.split(delim, x)[0]

    # remove titles
    x = re.sub(r"^\s*(dr|prof|mr|ms|mrs)\.?\s+", " ", x, flags=re.IGNORECASE)

    # tidy punctuation/whitespace
    x = re.sub(r"[^\w\s'.-]", " ", x)            # keep letters/digits/space/apostrophe/dot/hyphen
    x = re.sub(r"\s+", " ", x).strip()

    # accents -> ascii, title-case words
    x = strip_accents(x)
    x = " ".join(w.capitalize() for w in x.split())

    # length guard
    return x[:120].strip()

def final_clean(raw: str) -> str:
    if not isinstance(raw, str): return ""
    if raw in alias_map: return alias_map[raw]
    cleaned = basic_name_clean(raw)
    return alias_map.get(cleaned, cleaned)

# Build mapping from the *rawest* table (person-session)
raw_names = ps["person_name"].astype(str).fillna("")
freq = Counter(raw_names)
unique_raw = pd.Series(sorted(set(raw_names)))
cleaned = unique_raw.apply(final_clean)

def keyize(s: str) -> str:
    s2 = s.lower()
    s2 = re.sub(r"[^a-z]", "", s2)  # letters only for a coarse cluster key
    return s2

map_df = pd.DataFrame({
    "raw_name": unique_raw,
    "clean_name": cleaned,
})
map_df["raw_freq"] = map_df["raw_name"].map(freq)
map_df["cluster_key"] = map_df["clean_name"].apply(keyize)

# choose canonical per cluster (most frequent raw -> its cleaned)
canon_by_key = {}
for key, grp in map_df.groupby("cluster_key"):
    idx = grp["raw_freq"].idxmax()
    canon_by_key[key] = grp.loc[idx, "clean_name"]
map_df["canonical_name"] = map_df["cluster_key"].map(canon_by_key)

# Save mapping for review
map_df.sort_values(["cluster_key","clean_name","raw_freq"], ascending=[True, True, False]).to_csv(MAP_OUT, index=False)
print(f"[OK] Wrote name mapping: {MAP_OUT}")

# Helper to apply mapping (raw->canonical) with fallback to cleaner
raw_to_canon = dict(zip(map_df["raw_name"], map_df["canonical_name"]))

def apply_canonical(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["person_name"] = df["person_name"].map(raw_to_canon).fillna(df["person_name"].apply(final_clean))
    return df

# ========= APPLY TO PERSON-SESSION (and write) =========
ps_clean = apply_canonical(ps)
ps_clean.to_csv(PS_OUT, index=False)
print(f"[OK] Wrote cleaned person-session: {PS_OUT}  (rows={len(ps_clean)})")

# ========= RE-AGG LOGIC (keeps ints for counts, floats for means) =========
def build_agg_dict(df: pd.DataFrame):
    sum_cols = []
    mean_cols = []
    for c in df.columns:
        if c in ["person_name","conference","year","session_id","role_in_session","role_primary","team_ids"]:
            continue
        # counts / sums (discrete events) -> SUM
        if (c.endswith("_count") or c.endswith("_sum_score") or
            c in ["p_turns","p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
                  "sessions_total","facilitator","member","participant","unknown",
                  "teams_funded","teams_unfunded","teams_total",
                  "role_facilitator","role_nonfacilitator","role_on_team","role_in_funded",
                  "role_member","role_participant"]):
            sum_cols.append(c)
        # means / continuous -> MEAN
        elif (c.endswith("_mean_score") or c.endswith("_mean_score_weighted") or
              c.startswith("ctx_") or c in ["p_speaking_duration_sec"]):
            mean_cols.append(c)
        else:
            # default: if numeric and not a known count, treat as mean to be safe
            if pd.api.types.is_numeric_dtype(df[c]):
                mean_cols.append(c)
    agg = {**{c:"sum" for c in sum_cols}, **{c:"mean" for c in mean_cols}}
    return agg, sum_cols

def cast_int_counts(df: pd.DataFrame, count_cols):
    for c in count_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).round().astype(int)
    return df

# ========= RE-AGG PERSON–YEAR =========
py_clean = apply_canonical(py)
agg_py, sum_cols_py = build_agg_dict(py_clean)
py_final = (py_clean
            .groupby(["person_name","conference","year"], as_index=False)
            .agg(agg_py)
           )
py_final = cast_int_counts(py_final, sum_cols_py + ["year"])
py_final.to_csv(PY_OUT, index=False)
print(f"[OK] Wrote cleaned person–year: {PY_OUT}  (rows={len(py_final)})")

# ========= RE-AGG PERSON-LEVEL =========
p_clean = apply_canonical(p)
agg_p, sum_cols_p = build_agg_dict(p_clean)
p_final = (p_clean
           .groupby(["person_name"], as_index=False)
           .agg(agg_p)
          )
p_final = cast_int_counts(p_final, sum_cols_p)
p_final.to_csv(P_OUT, index=False)
print(f"[OK] Wrote cleaned person-level: {P_OUT}  (rows={len(p_final)})")

# ========= QUICK DIAGNOSTICS =========
sus = map_df[ (map_df["clean_name"].eq("")) | (~map_df["clean_name"].str.contains(r"\s")) ]
if not sus.empty:
    print("\n[Heads-up] Some names look empty or single-token; consider alias_map fixes. Examples:")
    print(sus.sort_values("raw_freq", ascending=False).head(15)[["raw_name","clean_name","canonical_name","raw_freq"]].to_string(index=False))
else:
    print("\n[OK] No obviously empty/single-token names in mapping.")

[OK] Wrote name mapping: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/name_clean_map.csv
[OK] Wrote cleaned person-session: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_session_CLEANED.csv  (rows=2073)
[OK] Wrote cleaned person–year: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year_FIXED_CLEANED.csv  (rows=771)
[OK] Wrote cleaned person-level: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/all_data_df-agg-ppl_FIXED_CLEANED.csv  (rows=637)

[Heads-up] Some names look empty or single-token; consider alias_map fixes. Examples:
                raw_name clean_name canonical_name  raw_freq
    Marie-Claire Arrieta      Marie          Marie         4
            John-Paul Yu       John           John         4
             Gang-yu Liu       Gang           Gang         4
   Anna-Karin Gustavsson       An

## rebuilding the analysis part 

In [11]:
# =========================================
# REBUILD ANALYSIS: Charts + Regressions + ROC (robust to collinearity)
# Requires: pandas, numpy, matplotlib, statsmodels, scikit-learn, xlsxwriter, tabulate
# =========================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.sm_exceptions import PerfectSeparationError
from patsy import dmatrices
import numpy.linalg as npl

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, classification_report
from sklearn.impute import SimpleImputer

# ------------------ CONFIG ------------------
DATA_CSV = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year.csv")
OUT_DIR  = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_rebuild")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS     = 5
INCLUDE_CTX  = True   # include ctx_* features along with p_* and ann_* in regressions and classifiers

# ------------------ LOAD ------------------
df = pd.read_csv(DATA_CSV)

# ---- Ensure role flags exist (derive if missing) ----
def ensure_role_flags(df):
    df = df.copy()

    # role_facilitator
    if "role_facilitator" not in df.columns:
        if "facilitator" in df.columns:
            df["role_facilitator"] = (pd.to_numeric(df["facilitator"], errors="coerce").fillna(0) > 0).astype(int)
        elif "role_primary" in df.columns:
            df["role_facilitator"] = (df["role_primary"].astype(str).str.lower() == "facilitator").astype(int)
        else:
            df["role_facilitator"] = 0

    # role_on_team
    if "role_on_team" not in df.columns:
        if "teams_total" in df.columns:
            df["role_on_team"] = (pd.to_numeric(df["teams_total"], errors="coerce").fillna(0) > 0).astype(int)
        elif "member" in df.columns:
            df["role_on_team"] = (pd.to_numeric(df["member"], errors="coerce").fillna(0) > 0).astype(int)
        else:
            df["role_on_team"] = 0

    # role_in_funded
    if "role_in_funded" not in df.columns:
        if "teams_funded" in df.columns:
            df["role_in_funded"] = (pd.to_numeric(df["teams_funded"], errors="coerce").fillna(0) > 0).astype(int)
        else:
            df["role_in_funded"] = 0

    for c in ["role_facilitator","role_on_team","role_in_funded"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    return df

df = ensure_role_flags(df)

# Which flags actually exist?
role_flags_available = [c for c in ["role_facilitator","role_on_team","role_in_funded"] if c in df.columns]

# ------------------ FEATURE SETS ------------------
# Person-level behavioral (transcript-derived)
p_feats   = [c for c in df.columns if c.startswith("p_") and pd.api.types.is_numeric_dtype(df[c])]
# Annotation categories
ann_feats = [c for c in df.columns if c.startswith("ann_") and pd.api.types.is_numeric_dtype(df[c])]
# Session-level context (if present)
ctx_feats = [c for c in df.columns if c.startswith("ctx_") and pd.api.types.is_numeric_dtype(df[c])]
if not INCLUDE_CTX:
    ctx_feats = []

num_feats = sorted(list(set(p_feats + ann_feats + ctx_feats)))

# Drop zero-variance / all-missing features
good_feats = []
for c in num_feats:
    s = pd.to_numeric(df[c], errors="coerce")
    if s.notna().any() and s.std(skipna=True) > 0:
        good_feats.append(c)
num_feats = sorted(good_feats)

print(f"Using {len(num_feats)} numeric features.")

# ------------------ CHARTS: Helper Plots ------------------
def bar_mean_with_ci(data, by_flag, metric, title, out_png):
    tmp = data[[by_flag, metric]].dropna()
    tmp[by_flag] = tmp[by_flag].astype(int)
    groups = tmp.groupby(by_flag)[metric]
    means = groups.mean()
    ns    = groups.size()
    sds   = groups.std()
    cis = 1.96 * sds / np.sqrt(ns.clip(lower=1))  # 95% CI

    fig, ax = plt.subplots(figsize=(5,4))
    ax.bar(["No","Yes"], [means.get(0, np.nan), means.get(1, np.nan)])
    ax.errorbar([0,1], [means.get(0, np.nan), means.get(1, np.nan)],
                yerr=[cis.get(0,0), cis.get(1,0)], fmt='none', capsize=4)
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.set_xlabel(by_flag)
    plt.tight_layout()
    fig.savefig(out_png, dpi=180)
    plt.close(fig)

def box_by_flag(data, by_flag, metric, title, out_png):
    tmp = data[[by_flag, metric]].dropna()
    tmp[by_flag] = tmp[by_flag].astype(int)
    fig, ax = plt.subplots(figsize=(5,4))
    ax.boxplot([tmp.loc[tmp[by_flag]==0, metric], tmp.loc[tmp[by_flag]==1, metric]],
               labels=["No","Yes"], showfliers=False)
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.set_xlabel(by_flag)
    plt.tight_layout()
    fig.savefig(out_png, dpi=180)
    plt.close(fig)

# Pick a concise subset of metrics to visualize (add more if you like)
bar_metrics = [
    "p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
    "ann_knowledge_sharing_count","ann_evaluation_practices_count",
    "ann_coordination_and_decision_practices_count","ann_relational_climate_count",
    "ann_participation_dynamics_count"
]
box_metrics = ["p_speaking_duration_sec","p_turns"]

# Make charts for each contrast flag that exists
for flag in role_flags_available:
    for m in bar_metrics:
        if m in df.columns:
            fn = OUT_DIR / f"bar_{m}_{flag}.png"
            bar_mean_with_ci(df, flag, m, f"{m} by {flag}", fn)
    for m in box_metrics:
        if m in df.columns:
            fn = OUT_DIR / f"box_{m}_{flag}.png"
            box_by_flag(df, flag, m, f"{m} by {flag}", fn)

print("[OK] Wrote bar/box charts.")

# ------------------ REGRESSIONS (Logit with FE, full-rank safe) ------------------
def _drop_problem_cols(X_df):
    """Drop zero-variance, duplicate columns; if still rank-deficient, keep a full-rank subset via QR."""
    X = X_df.copy()

    # Drop all-constant / zero-variance
    keep = [c for c in X.columns if X[c].std(ddof=0) > 0]
    X = X[keep]

    # Drop exact-duplicate columns
    seen = {}
    dedup_keep = []
    for c in X.columns:
        key = tuple(np.asarray(X[c]).round(12))
        if key in seen:
            continue
        seen[key] = c
        dedup_keep.append(c)
    X = X[dedup_keep]

    # Full-rank via QR pivot if needed
    A = X.values
    rank = np.linalg.matrix_rank(A)
    if rank == A.shape[1]:
        return X

    Q, R, piv = npl.qr(A, mode='reduced', pivoting=True)
    indep_idx = piv[:rank]
    keep_cols = [X.columns[i] for i in indep_idx]
    return X[keep_cols]

def logit_with_fe(data, target, predictors, fe_conf=True, fe_year=True, add_sess=True):
    """
    Build design via patsy, drop collinear columns, fit Logit.
    If MLE fails (singular/complete separation), fall back to fit_regularized.
    Returns: (data_used, model_or_result, robust_or_none, formula_string)
    """
    # Build formula
    rhs = []
    if add_sess and "sessions_total" in data.columns:
        rhs.append("sessions_total")
    rhs += [f"`{x}`" if (" " in x or "-" in x) else x for x in predictors]
    if fe_conf and "conference" in data.columns:
        rhs.append("C(conference)")
    if fe_year and "year" in data.columns:
        rhs.append("C(year)")
    formula = f"{target} ~ " + " + ".join(rhs)

    # Design matrices
    try:
        y, X = dmatrices(formula, data=data, return_type="dataframe", NA_action="drop")
    except Exception as e:
        print(f"[SKIP] Patsy failed for {target}: {e}")
        return None, None, None, formula

    y_vec = np.asarray(y).ravel().astype(int)
    if np.unique(y_vec).size < 2:
        print(f"[SKIP] {target}: only one class after NA drop.")
        return None, None, None, formula

    # Hold intercept, clean the rest
    const = None
    if "Intercept" in X.columns:
        const = X["Intercept"].copy()
        X = X.drop(columns=["Intercept"])
    X = _drop_problem_cols(X)
    if const is not None:
        X.insert(0, "Intercept", const.values)

    # Fit (request robust SEs directly); fallback to regularized if needed
    try:
        model = sm.Logit(y_vec, X)
        result = model.fit(disp=0, cov_type="HC3")   # <-- robust SEs here
        robust = result                               # already robust
        used = pd.concat([y.reset_index(drop=True), X.reset_index(drop=True)], axis=1)
        return used, result, robust, formula
    except (np.linalg.LinAlgError, PerfectSeparationError) as e:
        print(f"[Info] MLE failed for {target} ({type(e).__name__}); trying regularized fit.")
        model = sm.Logit(y_vec, X)
        result = model.fit_regularized(alpha=1.0, L1_wt=0.5, disp=0)
        robust = None  # robust SEs not available here
        used = pd.concat([y.reset_index(drop=True), X.reset_index(drop=True)], axis=1)
        return used, result, robust, formula

def tidy_from_result(result, robust_or_none, model_label):
    params = (robust_or_none.params if robust_or_none is not None else result.params)
    bse    = (robust_or_none.bse    if robust_or_none is not None else result.bse)
    pvals  = (robust_or_none.pvalues if robust_or_none is not None else result.pvalues)
    conf   = (robust_or_none.conf_int() if robust_or_none is not None else result.conf_int())

    out = pd.DataFrame({
        "predictor": params.index,
        "coef": params.values,
        "std_err": bse.values,
        "p_value": pvals.values,
        "ci_low": conf[0].values,
        "ci_high": conf[1].values,
        "model": model_label,
        "n_obs": int(result.nobs),
        "llf": result.llf,
        "llnull": getattr(result, "llnull", np.nan),
        "pseudo_r2": (1 - (result.llf / result.llnull)) if getattr(result, "llnull", 0) not in (0, np.nan) else np.nan,
        "llr_pvalue": getattr(result, "llr_pvalue", np.nan)
    })
    return out

tidy_list = []
summaries = []

for tgt, desc in [
    ("role_facilitator","Facilitator vs Non-facilitator"),
    ("role_on_team","On-team vs Not-on-team"),
    ("role_in_funded","Funded vs Not-funded")
]:
    if tgt not in role_flags_available:
        continue
    used, res, rob, form = logit_with_fe(df, tgt, predictors=num_feats, fe_conf=True, fe_year=True, add_sess=True)
    if res is None:
        continue

    # summary text
    summ = [f"=== {desc} ({tgt}) ===", f"Formula: {form}", res.summary2().as_text()]
    if rob is None:
        summ.append("\nNote: Regularized fit used (robust SEs not available).")
    else:
        summ.append("\nRobust (HC3) SEs used in tidy table.")
    summaries.append("\n".join(summ))

    tidy_list.append(tidy_from_result(res, rob, desc))

# Save tidy combined and summaries
if tidy_list:
    tidy_df = pd.concat(tidy_list, ignore_index=True)
    tidy_csv = OUT_DIR / "regression_tidy_results.csv"
    tidy_df.to_csv(tidy_csv, index=False)

    with open(OUT_DIR / "regression_model_summaries.txt","w") as f:
        f.write("\n\n".join(summaries))

    # Nicely formatted per-model tables
    def tidy_to_display(df_tidy, model_name):
        sub = df_tidy[df_tidy["model"]==model_name].copy()
        sub.loc[sub["predictor"].str.startswith("C(conference)"), "predictor"] = sub["predictor"].str.replace("C(conference)","Conf_FE", regex=False)
        sub.loc[sub["predictor"].str.startswith("C(year)"), "predictor"] = sub["predictor"].str.replace("C(year)","Year_FE", regex=False)
        sub["Coef (SE)"] = sub["coef"].round(3).astype(str) + " (" + sub["std_err"].round(3).astype(str) + ")"
        sub["p"] = sub["p_value"].apply(lambda x: f"{x:.3g}")
        keep = ["predictor","Coef (SE)","ci_low","ci_high","p","n_obs","pseudo_r2","llr_pvalue"]
        sub = sub[keep].rename(columns={"predictor":"Term","ci_low":"CI 2.5%","ci_high":"CI 97.5%","p":"p-value","n_obs":"N","pseudo_r2":"Pseudo R2","llr_pvalue":"LLR p"})
        return sub

    tables = {}
    for model_name in tidy_df["model"].unique():
        disp = tidy_to_display(tidy_df, model_name)
        safe = "".join(ch for ch in model_name if ch.isalnum() or ch in "_- ").strip().replace(" ","_")
        disp.to_csv(OUT_DIR / f"reg_table_{safe}.csv", index=False)
        try:
            disp.to_markdown(OUT_DIR / f"reg_table_{safe}.md", index=False)
        except Exception:
            pass
        tables[model_name] = disp

    xlsx_path = OUT_DIR / "regression_tables_by_model.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as xw:
        for model_name, disp in tables.items():
            sheet = model_name[:31]
            disp.to_excel(xw, index=False, sheet_name=sheet)

    print(f"[OK] Regressions saved: {tidy_csv} and {xlsx_path}")
else:
    print("[SKIP] No regression models were fit (missing targets or single-class after filtering).")

# ------------------ MULTICOLLINEARITY (VIF on key behaviors) ------------------
vif_feats = [c for c in ["p_speaking_duration_sec","p_turns","p_interruptions_made","p_overlaps",
                         "p_screenshare_segments","p_smile_self_total","p_smile_other_total",
                         "p_nods_received"] if c in df.columns]
if vif_feats:
    X_vif = df[vif_feats].fillna(0)
    X_vif = sm.add_constant(X_vif)
    vif_table = pd.DataFrame({
        "feature": X_vif.columns,
        "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    })
    vif_table.to_csv(OUT_DIR / "vif_behaviors.csv", index=False)
    print("[OK] VIF table written.")
else:
    print("[SKIP] No VIF — none of the key behavior features found.")

# ------------------ PREDICTIVE: ROC Curves with behavior-only ------------------
targets = [(t, d) for (t, d) in [
    ("role_facilitator","Facilitator vs Non-facilitator"),
    ("role_on_team","On-team vs Not-on-team"),
    ("role_in_funded","Funded vs Not-funded")
] if t in role_flags_available]

X_cols = num_feats  # p_, ann_, and (ctx_ if INCLUDE_CTX)
if targets and X_cols:
    numeric_transform = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("scale", StandardScaler())
    ])
    preprocess = ColumnTransformer(transformers=[("num", numeric_transform, X_cols)], remainder="drop")

    logit = LogisticRegression(
        penalty="l2",
        solver="liblinear",
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE
    )
    clf = Pipeline(steps=[("prep", preprocess), ("clf", logit)])
    cv  = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    summary_rows = []
    for target, desc in targets:
        dsub = df.dropna(subset=[target]).copy()
        if dsub[target].nunique() < 2:
            print(f"[SKIP] ROC for {target}: single class present.")
            continue

        y = dsub[target].astype(int).values
        X = dsub[X_cols]

        # CV probs + ROC
        y_prob = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:,1]
        fpr, tpr, _ = roc_curve(y, y_prob)
        roc_auc = auc(fpr, tpr)

        # Classification report at 0.5
        y_hat = (y_prob >= 0.5).astype(int)
        cr = classification_report(y, y_hat, output_dict=True, zero_division=0)
        pd.DataFrame(cr).to_csv(OUT_DIR / f"clf_classification_report_{target}.csv")

        # CV metric summary
        scoring = {"roc_auc": "roc_auc","accuracy":"accuracy","precision":"precision","recall":"recall","f1":"f1"}
        cv_res = cross_validate(clf, X, y, cv=cv, scoring=scoring, return_train_score=False, n_jobs=-1)
        row = {"target": target, "description": desc}
        for m in scoring:
            row[f"{m}_mean"] = float(cv_res[f"test_{m}"].mean())
            row[f"{m}_sd"]   = float(cv_res[f"test_{m}"].std())
        summary_rows.append(row)

        # ROC plot
        fig, ax = plt.subplots(figsize=(5,4))
        ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
        ax.plot([0,1],[0,1], linestyle="--")
        ax.set_title(f"ROC — {desc}")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.legend(loc="lower right")
        plt.tight_layout()
        fig.savefig(OUT_DIR / f"clf_roc_{target}.png", dpi=180)
        plt.close(fig)

    if summary_rows:
        summary_df = pd.DataFrame(summary_rows).sort_values("target")
        summary_df.to_csv(OUT_DIR / "clf_results_summary.csv", index=False)
        print("[OK] Predictive ROC + summaries written.")
else:
    print("[SKIP] ROC — no targets available or no numeric features.")

print("\nOutputs written to:", OUT_DIR)

Using 33 numeric features.


/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_68986/346889039.py:124: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([tmp.loc[tmp[by_flag]==0, metric], tmp.loc[tmp[by_flag]==1, metric]],
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_68986/346889039.py:124: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([tmp.loc[tmp[by_flag]==0, metric], tmp.loc[tmp[by_flag]==1, metric]],
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_68986/346889039.py:124: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([tmp.loc[tmp[by_flag]==0, metric], tmp.loc[tmp[by_flag]==1, metri

[OK] Wrote bar/box charts.
[Info] MLE failed for role_facilitator (LinAlgError); trying regularized fit.
[SKIP] role_in_funded: only one class after NA drop.
[OK] Regressions saved: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_rebuild/regression_tidy_results.csv and /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_rebuild/regression_tables_by_model.xlsx
[OK] VIF table written.
[SKIP] ROC for role_in_funded: single class present.
[OK] Predictive ROC + summaries written.

Outputs written to: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_rebuild
